# Stage 5 — XGBoost Challenger Development

This Notebook performs Development-only Challenger construction and selection. It replays the frozen 24,000/6,000 split, uses Holdout only for row/ID/hash identity checks, and deletes the Holdout object before any predictive work. All model selection uses one fixed five-fold Development partition. No Holdout target rate, predictor distribution, transformation, prediction, AUC, or KS is accessed here.

The output is exactly one frozen 26-conceptual-predictor / 29-column XGBoost specification, one unified CV-derived boosting-round count, a Pre-Holdout Freeze Record, and a full 24,000-row refit. Notebook 12 owns the one-time post-freeze Holdout comparison.


In [1]:
from pathlib import Path
from itertools import product
import hashlib
import importlib.metadata as metadata
import json
import math
import sys
import time

import numpy as np
import pandas as pd
import xgboost as xgb
from IPython.display import display
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold, train_test_split

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 1000)
pd.set_option("display.width", 280)
pd.set_option("display.float_format", lambda value: f"{value:.9f}")

TARGET = "default.payment.next.month"
ID_COL = "ID"
RANDOM_STATE = 42
EXPECTED_TRAIN_HASH = "bcb37af63ae7005df78e6140837594b5d650d27148b46b5d51354430ad958857"
EXPECTED_TEST_HASH = "d6748249f9f9fb2855746e0ed980b5013047267104090a14aa8138a700fe6f0f"
EXPECTED_ROOT = Path("/Users/yu/2.工作/Risk Analysis Project")
MAX_BOOST_ROUNDS = 2000
EARLY_STOPPING_PATIENCE = 50
CV_FOLDS = 5
AUC_SD_DDOF = 1

PAY_COLS = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]
BILL_COLS = [f"BILL_AMT{i}" for i in range(1, 7)]
PAY_AMT_COLS = [f"PAY_AMT{i}" for i in range(1, 7)]
EDUCATION_CATEGORIES = ["Graduate School", "University", "High School", "Other / Unknown"]
EDUCATION_MAP = {
    1: "Graduate School", 2: "University", 3: "High School",
    0: "Other / Unknown", 4: "Other / Unknown", 5: "Other / Unknown", 6: "Other / Unknown",
}
CONCEPTUAL_PREDICTORS = [
    *PAY_COLS,
    "delinquent_months", "max_delinquency", "severe_delinquent_months",
    "EDUCATION_GROUP", "LIMIT_BAL",
    *BILL_COLS,
    *PAY_AMT_COLS,
    "current_utilization", "avg_utilization", "max_utilization",
]
EDUCATION_INDICATORS = [f"EDUCATION_GROUP__{category}" for category in EDUCATION_CATEGORIES]
ENCODED_COLUMNS = [
    *PAY_COLS,
    "delinquent_months", "max_delinquency", "severe_delinquent_months",
    *EDUCATION_INDICATORS, "LIMIT_BAL",
    *BILL_COLS, *PAY_AMT_COLS,
    "current_utilization", "avg_utilization", "max_utilization",
]
EXCLUDED_VARIABLES = ["SEX", "MARRIAGE_GROUP", "AGE"]

PRIMARY_PREDICTORS = [
    "PAY_0", "severe_delinquent_months",
    "PAY_AMT1", "PAY_AMT2", "PAY_AMT3", "PAY_AMT4", "PAY_AMT5", "PAY_AMT6",
    "LIMIT_BAL", "avg_utilization",
]
PRIMARY_REFERENCE = pd.Series({
    "const": -1.254276175288,
    "PAY_0": -0.556303701580,
    "severe_delinquent_months": -0.513060045720,
    "PAY_AMT1": -0.193755569096,
    "PAY_AMT2": -0.222092687932,
    "PAY_AMT3": -0.245401438534,
    "PAY_AMT4": -0.165041793818,
    "PAY_AMT5": -0.097122437149,
    "PAY_AMT6": -0.164349101704,
    "LIMIT_BAL": -0.226160642201,
    "avg_utilization": -0.405621157758,
})
PRIMARY_REFERENCE_AUC = 0.772417097808
PRIMARY_REFERENCE_KS = 0.430307867280

assert len(CONCEPTUAL_PREDICTORS) == len(set(CONCEPTUAL_PREDICTORS)) == 26
assert len(ENCODED_COLUMNS) == len(set(ENCODED_COLUMNS)) == 29
assert PRIMARY_REFERENCE.index.tolist() == ["const", *PRIMARY_PREDICTORS]

def locate_project_root():
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / "data/raw/UCI_Credit_Card.csv").is_file():
            return candidate
    raise FileNotFoundError("Formal raw CSV not found from launch path or parents")

def stable_id_hash(values):
    canonical = "\n".join(str(int(value)) for value in sorted(values))
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()

project_root = locate_project_root()
assert project_root == EXPECTED_ROOT
data_path = project_root / "data/raw/UCI_Credit_Card.csv"
environment = pd.DataFrame({"Value": [
    sys.executable, sys.version.split()[0], metadata.version("xgboost"),
    metadata.version("numpy"), metadata.version("pandas"), metadata.version("scipy"),
    metadata.version("scikit-learn"),
]}, index=["Python executable", "Python", "XGBoost", "NumPy", "pandas", "SciPy", "scikit-learn"])
assert Path(sys.executable).resolve() == (project_root / ".venv/bin/python").resolve()
assert sys.version.split()[0] == "3.9.6" and xgb.__version__ == "2.1.4"
display(environment)
print("Conceptual predictors (26):", CONCEPTUAL_PREDICTORS)
print("Encoded columns (29):", ENCODED_COLUMNS)


,Value
Python executable,/Users/yu/2.工作/Risk Analysis Project/.venv/bin...
Python,3.9.6
XGBoost,2.1.4
NumPy,2.0.2
pandas,2.3.3
SciPy,1.13.1
scikit-learn,1.6.1


Conceptual predictors (26): ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'delinquent_months', 'max_delinquency', 'severe_delinquent_months', 'EDUCATION_GROUP', 'LIMIT_BAL', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'current_utilization', 'avg_utilization', 'max_utilization']
Encoded columns (29): ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'delinquent_months', 'max_delinquency', 'severe_delinquent_months', 'EDUCATION_GROUP__Graduate School', 'EDUCATION_GROUP__University', 'EDUCATION_GROUP__High School', 'EDUCATION_GROUP__Other / Unknown', 'LIMIT_BAL', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'current_utilization', 'avg_utilization', 'max_utilization']


## 1. Frozen outer split replay and immediate Holdout sealing

The raw file is read only to replay the single approved stratified 80/20 split. Before deletion, the Holdout object is used only for row count, ID overlap/coverage, and the frozen ID hash. No Holdout label distribution or predictor value is inspected.


In [2]:
raw_df = pd.read_csv(data_path)
assert raw_df.shape == (30000, 25) and raw_df[ID_COL].is_unique

development_raw, sealed_holdout = train_test_split(
    raw_df,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=raw_df[TARGET],
)
development_ids = set(development_raw[ID_COL].astype(int))
holdout_ids = set(sealed_holdout[ID_COL].astype(int))
full_ids = set(raw_df[ID_COL].astype(int))
train_hash = stable_id_hash(development_ids)
test_hash = stable_id_hash(holdout_ids)
id_overlap = len(development_ids & holdout_ids)
id_union_complete = development_ids | holdout_ids == full_ids

assert len(development_raw) == 24000 and len(sealed_holdout) == 6000
assert id_overlap == 0 and id_union_complete
print("Computed Train ID SHA-256:", train_hash)
print("Computed Test ID SHA-256:", test_hash)
assert train_hash == EXPECTED_TRAIN_HASH and test_hash == EXPECTED_TEST_HASH

split_evidence = pd.DataFrame({"Actual": [
    len(raw_df), len(development_raw), len(sealed_holdout), id_overlap,
    id_union_complete, train_hash, test_hash,
]}, index=[
    "Raw rows", "Development rows", "Holdout rows (identity only)", "ID overlap",
    "Full ID union coverage", "Train ID SHA-256", "Test ID SHA-256",
])
display(split_evidence)

del sealed_holdout, holdout_ids, full_ids, raw_df
HOLDOUT_SEAL_STATUS = (
    "SEALED — Holdout object deleted after row count, ID overlap/coverage, and frozen ID hash checks; "
    "no Holdout target or predictor analysis"
)
print(HOLDOUT_SEAL_STATUS)

y_dev = development_raw[TARGET].astype(int).copy()
development_good = int((y_dev == 0).sum())
development_bad = int((y_dev == 1).sum())
assert (development_good, development_bad) == (18691, 5309)
print(f"Development target: Good={development_good:,}; Bad={development_bad:,}; Bad rate={y_dev.mean():.9%}")


Computed Train ID SHA-256: bcb37af63ae7005df78e6140837594b5d650d27148b46b5d51354430ad958857
Computed Test ID SHA-256: d6748249f9f9fb2855746e0ed980b5013047267104090a14aa8138a700fe6f0f


,Actual
Raw rows,30000
Development rows,24000
Holdout rows (identity only),6000
ID overlap,0
Full ID union coverage,True
Train ID SHA-256,bcb37af63ae7005df78e6140837594b5d650d27148b46b...
Test ID SHA-256,d6748249f9f9fb2855746e0ed980b5013047267104090a...


SEALED — Holdout object deleted after row count, ID overlap/coverage, and frozen ID hash checks; no Holdout target or predictor analysis
Development target: Good=18,691; Bad=5,309; Bad rate=22.120833333%


## 2. Frozen Stage 3 feature definitions and explicit education encoding

The six deterministic Stage 3 features are reproduced without modification. Negative bill amounts are floored only inside utilization calculations; utilization above 1 remains uncapped. `EDUCATION_GROUP` uses the frozen raw-code mapping and expands to all four explicit indicator columns—no reference category is dropped and no category universe is learned from Holdout.


In [3]:
def apply_frozen_stage3_features(raw_predictor_input):
    assert TARGET not in raw_predictor_input.columns
    required = [ID_COL, "EDUCATION", "LIMIT_BAL", *PAY_COLS, *BILL_COLS, *PAY_AMT_COLS]
    source_before = raw_predictor_input[required].copy(deep=True)
    assert set(raw_predictor_input["EDUCATION"].unique()).issubset(EDUCATION_MAP)
    assert (raw_predictor_input["LIMIT_BAL"] > 0).all()

    work = raw_predictor_input.copy(deep=True)
    work["EDUCATION_GROUP"] = work["EDUCATION"].map(EDUCATION_MAP)
    assert work["EDUCATION_GROUP"].notna().all()
    raw_pay = work[PAY_COLS]
    work["delinquent_months"] = (raw_pay > 0).sum(axis=1).astype("int64")
    work["max_delinquency"] = raw_pay.clip(lower=0).max(axis=1).astype("int64")
    work["severe_delinquent_months"] = (raw_pay >= 2).sum(axis=1).astype("int64")
    positive_bills = work[BILL_COLS].clip(lower=0)
    work["current_utilization"] = positive_bills["BILL_AMT1"] / work["LIMIT_BAL"]
    work["avg_utilization"] = positive_bills.mean(axis=1) / work["LIMIT_BAL"]
    work["max_utilization"] = positive_bills.max(axis=1) / work["LIMIT_BAL"]
    pd.testing.assert_frame_equal(work[required], source_before, check_exact=True)
    return work

def build_challenger_matrix(feature_frame):
    matrix = pd.DataFrame(index=feature_frame.index)
    for name in [*PAY_COLS, "delinquent_months", "max_delinquency", "severe_delinquent_months"]:
        matrix[name] = feature_frame[name]
    for category, column in zip(EDUCATION_CATEGORIES, EDUCATION_INDICATORS):
        matrix[column] = (feature_frame["EDUCATION_GROUP"] == category).astype("int8")
    matrix["LIMIT_BAL"] = feature_frame["LIMIT_BAL"]
    for name in [*BILL_COLS, *PAY_AMT_COLS, "current_utilization", "avg_utilization", "max_utilization"]:
        matrix[name] = feature_frame[name]
    assert matrix.columns.tolist() == ENCODED_COLUMNS
    assert matrix.shape[1] == 29
    assert matrix[EDUCATION_INDICATORS].sum(axis=1).eq(1).all()
    assert matrix.notna().all().all() and np.isfinite(matrix.to_numpy(dtype=float)).all()
    return matrix.astype(float)

development_features = apply_frozen_stage3_features(development_raw.drop(columns=[TARGET]))
X_dev = build_challenger_matrix(development_features)
assert X_dev.shape == (24000, 29)
assert X_dev.index.equals(development_raw.index) and y_dev.index.equals(X_dev.index)
assert not any(name in X_dev.columns for name in EXCLUDED_VARIABLES)
assert not any("WOE" in name.upper() for name in X_dev.columns)

education_audit = pd.DataFrame({
    "Raw codes": ["1", "2", "3", "0,4,5,6"],
    "Frozen group": EDUCATION_CATEGORIES,
    "Encoded column": EDUCATION_INDICATORS,
    "Development count": [int((development_features["EDUCATION_GROUP"] == c).sum()) for c in EDUCATION_CATEGORIES],
})
display(education_audit)
display(pd.DataFrame({"Actual": [
    len(CONCEPTUAL_PREDICTORS), X_dev.shape, X_dev.isna().sum().sum(),
    int((~np.isfinite(X_dev.to_numpy())).sum()), EXCLUDED_VARIABLES,
]}, index=["Conceptual predictor count", "Development matrix shape", "Missing values", "Non-finite values", "Explicit exclusions"]))


,Raw codes,Frozen group,Encoded column,Development count
0,1,Graduate School,EDUCATION_GROUP__Graduate School,8455
1,2,University,EDUCATION_GROUP__University,11256
2,3,High School,EDUCATION_GROUP__High School,3903
3,"0,4,5,6",Other / Unknown,EDUCATION_GROUP__Other / Unknown,386


,Actual
Conceptual predictor count,26
Development matrix shape,"(24000, 29)"
Missing values,0
Non-finite values,0
Explicit exclusions,"[SEX, MARRIAGE_GROUP, AGE]"


## 3. Frozen Stage 4.7 Primary reproduction on Development only

The approved feature definitions, frozen bins, and Full-Train WOE convention `ln(DistGood / DistBad)` are reconstructed from Development only. The reviewed ten-variable unpenalized-MLE coefficient vector is loaded from the frozen Stage 4.7 specification without refitting, and its apparent Development AUC/KS are reproduced. This is a numerical reproduction gate, not Primary redevelopment.


In [4]:
PAY_DOMAIN = set(range(-2, 9))
PAY0_MAP = {
    -2: "[-2,-1,0]", -1: "[-2,-1,0]", 0: "[-2,-1,0]", 1: "[1]",
    2: "[2+]", 3: "[2+]", 4: "[2+]", 5: "[2+]", 6: "[2+]", 7: "[2+]", 8: "[2+]",
}

def finite_number(value, name):
    if pd.isna(value) or not isinstance(value, (int, float, np.integer, np.floating)):
        raise ValueError(f"{name}: invalid input")
    numeric = float(value)
    if not np.isfinite(numeric):
        raise ValueError(f"{name}: non-finite input")
    return numeric

def map_pay_0(value):
    numeric = finite_number(value, "PAY_0")
    raw = int(numeric)
    if raw != numeric or raw not in PAY_DOMAIN:
        raise ValueError("PAY_0 outside approved domain")
    return PAY0_MAP[raw]

def map_severe(value):
    numeric = finite_number(value, "severe_delinquent_months")
    raw = int(numeric)
    mapping = {0: "0", 1: "1", 2: "2", 3: "3", 4: "4+", 5: "4+", 6: "4+"}
    if raw != numeric or raw not in mapping:
        raise ValueError("severe_delinquent_months outside approved domain")
    return mapping[raw]

def map_payment(value):
    numeric = finite_number(value, "PAY_AMT")
    if numeric < 0:
        raise ValueError("negative PAY_AMT")
    if numeric == 0:
        return "0"
    if numeric <= 5000:
        return "(0,5k]"
    if numeric <= 20000:
        return "(5k,20k]"
    return ">20k"

def map_limit(value):
    numeric = finite_number(value, "LIMIT_BAL")
    if numeric <= 30000:
        return "<=30k"
    if numeric <= 140000:
        return "(30k,140k]"
    if numeric <= 240000:
        return "(140k,240k]"
    return ">240k"

def map_utilization(value):
    numeric = finite_number(value, "avg_utilization")
    if numeric < 0:
        raise ValueError("negative utilization")
    if numeric == 0:
        return "0"
    if numeric <= 0.20:
        return "(0,0.2]"
    if numeric <= 0.60:
        return "(0.2,0.6]"
    if numeric <= 1:
        return "(0.6,1]"
    return ">1"

BIN_ORDERS = {
    "PAY_0": ["[-2,-1,0]", "[1]", "[2+]"],
    "severe_delinquent_months": ["0", "1", "2", "3", "4+"],
    **{name: ["0", "(0,5k]", "(5k,20k]", ">20k"] for name in PAY_AMT_COLS},
    "LIMIT_BAL": ["<=30k", "(30k,140k]", "(140k,240k]", ">240k"],
    "avg_utilization": ["0", "(0,0.2]", "(0.2,0.6]", "(0.6,1]", ">1"],
}
BIN_MAPPERS = {
    "PAY_0": map_pay_0,
    "severe_delinquent_months": map_severe,
    **{name: map_payment for name in PAY_AMT_COLS},
    "LIMIT_BAL": map_limit,
    "avg_utilization": map_utilization,
}
assert list(BIN_ORDERS) == list(BIN_MAPPERS) == PRIMARY_PREDICTORS

primary_frame = development_features.copy(deep=True)
primary_bins = pd.DataFrame(index=primary_frame.index)
for predictor in PRIMARY_PREDICTORS:
    labels = primary_frame[predictor].map(BIN_MAPPERS[predictor])
    primary_bins[predictor] = pd.Categorical(labels, categories=BIN_ORDERS[predictor], ordered=True)
    assert primary_bins[predictor].notna().all()

primary_woe = pd.DataFrame(index=primary_frame.index)
primary_woe_rows = []
for predictor in PRIMARY_PREDICTORS:
    grouped = pd.DataFrame({"Bin": primary_bins[predictor], TARGET: y_dev}).groupby("Bin", observed=False)[TARGET].agg(Count="size", Bad="sum")
    grouped = grouped.reindex(BIN_ORDERS[predictor], fill_value=0)
    grouped["Good"] = grouped["Count"] - grouped["Bad"]
    assert (grouped["Count"] > 0).all() and (grouped[["Good", "Bad"]] > 0).all().all()
    dist_good = grouped["Good"] / grouped["Good"].sum()
    dist_bad = grouped["Bad"] / grouped["Bad"].sum()
    woe = np.log(dist_good / dist_bad)
    mapping = woe.to_dict()
    primary_woe[predictor] = primary_bins[predictor].map(mapping).astype(float)
    for key in BIN_ORDERS[predictor]:
        primary_woe_rows.append({"Predictor": predictor, "Bin": key, "Count": int(grouped.loc[key, "Count"]), "Good": int(grouped.loc[key, "Good"]), "Bad": int(grouped.loc[key, "Bad"]), "WOE": float(woe.loc[key])})

assert primary_woe.shape == (24000, 10) and len(primary_woe_rows) == 41
assert primary_woe.notna().all().all() and np.isfinite(primary_woe.to_numpy()).all()

primary_params = PRIMARY_REFERENCE.copy()
primary_linear_predictor = (
    float(primary_params["const"])
    + primary_woe.to_numpy(dtype=float) @ primary_params[PRIMARY_PREDICTORS].to_numpy(dtype=float)
)
primary_probability = 1.0 / (1.0 + np.exp(-primary_linear_predictor))
primary_fpr, primary_tpr, primary_thresholds = roc_curve(y_dev, primary_probability, pos_label=1)
primary_auc = float(roc_auc_score(y_dev, primary_probability))
primary_ks = float(np.max(primary_tpr - primary_fpr))
primary_param_diff = (primary_params - PRIMARY_REFERENCE).abs()
primary_intercept_diff = float(primary_param_diff["const"])
primary_max_coefficient_diff = float(primary_param_diff[PRIMARY_PREDICTORS].max())
primary_auc_diff = abs(primary_auc - PRIMARY_REFERENCE_AUC)
primary_ks_diff = abs(primary_ks - PRIMARY_REFERENCE_KS)
assert primary_intercept_diff <= 1e-9
assert primary_max_coefficient_diff <= 1e-9
assert primary_auc_diff <= 1e-9 and primary_ks_diff <= 1e-9

primary_reproduction = pd.DataFrame({"Actual": [
    float(primary_params["const"]), primary_intercept_diff, primary_max_coefficient_diff,
    primary_auc, primary_auc_diff, primary_ks, primary_ks_diff,
]}, index=[
    "Intercept", "Intercept absolute difference", "Maximum coefficient absolute difference",
    "Apparent Development AUC", "AUC absolute difference", "Apparent Development KS", "KS absolute difference",
])
display(primary_reproduction)
print("PASS — frozen Stage 4.7 Primary loaded without refit and its WOE predictions/AUC/KS reproduced within absolute tolerance 1e-9; PAY_AMT5 retained.")


,Actual
Intercept,-1.254276175
Intercept absolute difference,0.000000000
Maximum coefficient absolute difference,0.000000000
Apparent Development AUC,0.772417098
AUC absolute difference,0.000000000
Apparent Development KS,0.430307867
KS absolute difference,0.000000000


PASS — frozen Stage 4.7 Primary loaded without refit and its WOE predictions/AUC/KS reproduced within absolute tolerance 1e-9; PAY_AMT5 retained.


## 4. One deterministic five-fold Development partition

`StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` is generated once and reused for the baseline, all grouped searches, and the Final Joint Grid. Every Development observation is in validation exactly once; no Holdout row enters CV.


In [5]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_splits = list(skf.split(np.zeros(len(X_dev)), y_dev.to_numpy()))
assert len(fold_splits) == 5
validation_counts = np.zeros(len(X_dev), dtype=int)
validation_id_sets = []
fold_audit_rows = []

for fold, (train_pos, val_pos) in enumerate(fold_splits, start=1):
    validation_counts[val_pos] += 1
    train_ids_fold = set(development_raw.iloc[train_pos][ID_COL].astype(int))
    val_ids_fold = set(development_raw.iloc[val_pos][ID_COL].astype(int))
    validation_id_sets.append(val_ids_fold)
    assert train_ids_fold.isdisjoint(val_ids_fold)
    y_train_fold = y_dev.iloc[train_pos]
    y_val_fold = y_dev.iloc[val_pos]
    fold_audit_rows.append({
        "Fold": fold,
        "Fold-Train N": len(train_pos),
        "Validation N": len(val_pos),
        "Validation Good": int((y_val_fold == 0).sum()),
        "Validation Bad": int((y_val_fold == 1).sum()),
        "Validation Bad Rate": float(y_val_fold.mean()),
        "Validation ID SHA-256": stable_id_hash(val_ids_fold),
    })

assert np.all(validation_counts == 1)
assert set.union(*validation_id_sets) == development_ids
assert all(validation_id_sets[i].isdisjoint(validation_id_sets[j]) for i in range(5) for j in range(i + 1, 5))
fold_audit = pd.DataFrame(fold_audit_rows)
display(fold_audit)
print("PASS — validation folds pairwise disjoint; union=24,000 Development IDs; each ID validates exactly once.")

ddev = xgb.DMatrix(X_dev, label=y_dev, feature_names=ENCODED_COLUMNS)
FOLD_DATA = []
for train_pos, val_pos in fold_splits:
    FOLD_DATA.append((
        xgb.DMatrix(X_dev.iloc[train_pos], label=y_dev.iloc[train_pos], feature_names=ENCODED_COLUMNS),
        xgb.DMatrix(X_dev.iloc[val_pos], label=y_dev.iloc[val_pos], feature_names=ENCODED_COLUMNS),
        y_dev.iloc[val_pos].to_numpy(),
    ))


,Fold,Fold-Train N,Validation N,Validation Good,Validation Bad,Validation Bad Rate,Validation ID SHA-256
0,1,19200,4800,3739,1061,0.221041667,77aa267a9e7aa92775407bf611fa544bfda52bc15dfdba...
1,2,19200,4800,3738,1062,0.221250000,9c90b6d13f3e4c963fbf131fa658591abcfe676719bc18...
2,3,19200,4800,3738,1062,0.221250000,5b835279e92cb56f1c4d77ec01d4d9e0cba39c0d2f3f07...
3,4,19200,4800,3738,1062,0.221250000,1b474e892452313444dcb2f51811d8db2d96fda84cc0ef...
4,5,19200,4800,3738,1062,0.221250000,2933e9d11716a8bfa294057f2d24a1f39938d6736fb2f4...


PASS — validation folds pairwise disjoint; union=24,000 Development IDs; each ID validates exactly once.


## 5. XGBoost 2.1.4 defaults, deterministic settings, and metric engine

The untuned baseline uses XGBoost's relevant default model values (`100` rounds, depth `6`, child weight `1`, learning rate `0.3`, full row/column sampling, zero gamma/alpha, lambda `1`) plus explicit deterministic/runtime settings. `tree_method='hist'`, `seed=42`, and `nthread=4` are fixed across all candidates. Class weighting remains neutral (`scale_pos_weight=1`).

For every tuned combination, `xgboost.cv` receives the same explicit fold membership and selects one unified round from the aggregate five-fold validation-AUC curve with at most 2,000 rounds and patience 50. Fold models are then fitted at exactly that unified round to obtain five AUC/KS values. Reported SD is the accepted sample SD (`ddof=1`); KS is `max(TPR-FPR)` for positive class 1.


In [6]:
BASE_PARAMS = {
    "objective": "binary:logistic",
    "booster": "gbtree",
    "eval_metric": "auc",
    "tree_method": "hist",
    "eta": 0.30,
    "max_depth": 6,
    "min_child_weight": 1,
    "subsample": 1.0,
    "colsample_bytree": 1.0,
    "gamma": 0.0,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "scale_pos_weight": 1.0,
    "seed": 42,
    "nthread": 4,
    "verbosity": 0,
}

default_probe = xgb.train(BASE_PARAMS, ddev, num_boost_round=1, verbose_eval=False)
default_config = json.loads(default_probe.save_config())
tree_defaults = default_config["learner"]["gradient_booster"]["tree_train_param"]
default_evidence = pd.DataFrame({"Actual": [
    xgb.__version__, xgb.XGBClassifier().get_num_boosting_rounds(),
    tree_defaults["max_depth"], tree_defaults["min_child_weight"], tree_defaults["eta"],
    tree_defaults["subsample"], tree_defaults["colsample_bytree"], tree_defaults["gamma"],
    tree_defaults["reg_alpha"], tree_defaults["reg_lambda"],
    BASE_PARAMS["tree_method"], BASE_PARAMS["seed"], BASE_PARAMS["nthread"], BASE_PARAMS["scale_pos_weight"],
]}, index=[
    "XGBoost version", "Default sklearn boosting rounds", "max_depth", "min_child_weight", "learning_rate",
    "subsample", "colsample_bytree", "gamma", "reg_alpha", "reg_lambda",
    "Explicit tree_method", "Explicit seed", "Explicit nthread", "Neutral scale_pos_weight",
])
display(default_evidence)
del default_probe

def ks_metric(y_true, probability):
    fpr, tpr, thresholds = roc_curve(y_true, probability, pos_label=1)
    gaps = tpr - fpr
    idx = int(np.argmax(gaps))
    return float(gaps[idx]), float(thresholds[idx])

def fold_metrics_at_round(params, best_round):
    rows = []
    for fold, (dtrain_fold, dval_fold, y_val_fold) in enumerate(FOLD_DATA, start=1):
        booster = xgb.train(params, dtrain_fold, num_boost_round=int(best_round), verbose_eval=False)
        probability = booster.predict(dval_fold)
        assert len(probability) == len(y_val_fold)
        assert np.isfinite(probability).all() and ((probability >= 0) & (probability <= 1)).all()
        auc = float(roc_auc_score(y_val_fold, probability))
        ks, threshold = ks_metric(y_val_fold, probability)
        rows.append({"Fold": fold, "AUC": auc, "KS": ks, "KS threshold (diagnostic)": threshold})
    return pd.DataFrame(rows)

def result_row(label, params, best_round, metrics, aggregate_cv_auc=None, elapsed=None):
    auc_values = metrics["AUC"].to_numpy(float)
    ks_values = metrics["KS"].to_numpy(float)
    row = {
        "Label": label,
        "max_depth": int(params["max_depth"]),
        "min_child_weight": float(params["min_child_weight"]),
        "subsample": float(params["subsample"]),
        "colsample_bytree": float(params["colsample_bytree"]),
        "learning_rate": float(params["eta"]),
        "best_round": int(best_round),
        **{f"Fold {i} AUC": float(v) for i, v in enumerate(auc_values, 1)},
        "Mean AUC": float(np.mean(auc_values)),
        "AUC SD": float(np.std(auc_values, ddof=AUC_SD_DDOF)),
        **{f"Fold {i} KS": float(v) for i, v in enumerate(ks_values, 1)},
        "Mean KS": float(np.mean(ks_values)),
        "KS SD": float(np.std(ks_values, ddof=AUC_SD_DDOF)),
        "Aggregate CV AUC at best round": float(aggregate_cv_auc) if aggregate_cv_auc is not None else float(np.mean(auc_values)),
        "Elapsed seconds": float(elapsed) if elapsed is not None else np.nan,
    }
    if aggregate_cv_auc is not None:
        assert abs(row["Mean AUC"] - row["Aggregate CV AUC at best round"]) <= 1e-8
    return row

def evaluate_tuned_combination(label, overrides):
    params = {**BASE_PARAMS, **overrides}
    started = time.time()
    boosters = [xgb.Booster(params, cache=[dtrain, dvalid]) for dtrain, dvalid, _ in FOLD_DATA]
    best_mean_auc = -np.inf
    best_round = 0
    best_fold_aucs = None
    rounds_without_improvement = 0

    for iteration in range(MAX_BOOST_ROUNDS):
        fold_aucs = []
        for booster, (dtrain, dvalid, _) in zip(boosters, FOLD_DATA):
            booster.update(dtrain, iteration)
            evaluation = booster.eval(dvalid, name="validation", iteration=iteration)
            fold_aucs.append(float(evaluation.rsplit(":", 1)[1]))
        mean_auc = float(np.mean(fold_aucs))
        if mean_auc > best_mean_auc:
            best_mean_auc = mean_auc
            best_round = iteration + 1
            best_fold_aucs = list(fold_aucs)
            rounds_without_improvement = 0
        else:
            rounds_without_improvement += 1
        if rounds_without_improvement >= EARLY_STOPPING_PATIENCE:
            break

    assert 1 <= best_round <= MAX_BOOST_ROUNDS
    metrics_rows = []
    for fold, (booster, (_, dvalid, y_valid), expected_auc) in enumerate(
        zip(boosters, FOLD_DATA, best_fold_aucs), start=1
    ):
        probability = booster.predict(dvalid, iteration_range=(0, best_round))
        assert np.isfinite(probability).all() and ((probability >= 0) & (probability <= 1)).all()
        auc = float(roc_auc_score(y_valid, probability))
        assert abs(auc - expected_auc) <= 1e-12
        ks, threshold = ks_metric(y_valid, probability)
        metrics_rows.append({
            "Fold": fold, "AUC": auc, "KS": ks, "KS threshold (diagnostic)": threshold,
        })
    metrics = pd.DataFrame(metrics_rows)
    row = result_row(
        label, params, best_round, metrics, best_mean_auc, time.time() - started
    )
    return row

def rank_table(table):
    return table.sort_values(
        ["Mean AUC", "AUC SD", "max_depth", "min_child_weight", "best_round"],
        ascending=[False, True, True, False, True],
        kind="mergesort",
    ).reset_index(drop=True)

def near_best(table):
    ranked = rank_table(table)
    best = ranked.iloc[0]
    se_best = float(best["AUC SD"] / math.sqrt(CV_FOLDS))
    threshold = float(best["Mean AUC"] - se_best)
    region = table.loc[table["Mean AUC"] >= threshold].copy()
    return best, se_best, threshold, region

def boundary_profile(table, parameter, tested_values, natural_upper=False):
    profile = table.groupby(parameter, sort=False)["Mean AUC"].max().reindex(tested_values)
    values = profile.to_numpy(float)
    flags = []
    if len(values) >= 2 and int(np.nanargmax(values)) == 0 and np.all(np.diff(values) < 0):
        flags.append(f"{parameter}: performance improves continuously toward tested lower boundary {tested_values[0]}")
    if (not natural_upper) and len(values) >= 2 and int(np.nanargmax(values)) == len(values) - 1 and np.all(np.diff(values) > 0):
        flags.append(f"{parameter}: performance improves continuously toward tested upper boundary {tested_values[-1]}")
    return profile.rename("Best Mean AUC").reset_index(), flags


,Actual
XGBoost version,2.1.4
Default sklearn boosting rounds,100
max_depth,6
min_child_weight,1
learning_rate,0.300000012
subsample,1
colsample_bytree,1
gamma,0
reg_alpha,0
reg_lambda,1


## 6. Untuned XGBoost baseline

The baseline uses exactly 100 boosting rounds, the fixed 29-column matrix, and the same five folds. It is a reference only and does not replace the required structured tuning protocol.


In [7]:
baseline_started = time.time()
baseline_fold_metrics = fold_metrics_at_round(BASE_PARAMS, 100)
baseline_row = result_row("Untuned baseline", BASE_PARAMS, 100, baseline_fold_metrics, elapsed=time.time() - baseline_started)
baseline_results = pd.DataFrame([baseline_row])
display(baseline_fold_metrics)
display(baseline_results)


,Fold,AUC,KS,KS threshold (diagnostic)
0,1,0.768780884,0.406067285,0.251641452
1,2,0.761869621,0.396033409,0.235833496
2,3,0.750501920,0.380399198,0.230375916
3,4,0.771964826,0.407183212,0.236243948
4,5,0.762276825,0.397871305,0.206026003


,Label,max_depth,min_child_weight,subsample,colsample_bytree,learning_rate,best_round,Fold 1 AUC,Fold 2 AUC,Fold 3 AUC,Fold 4 AUC,Fold 5 AUC,Mean AUC,AUC SD,Fold 1 KS,Fold 2 KS,Fold 3 KS,Fold 4 KS,Fold 5 KS,Mean KS,KS SD,Aggregate CV AUC at best round,Elapsed seconds
0,Untuned baseline,6,1.000000000,1.000000000,1.000000000,0.300000000,100,0.768780884,0.761869621,0.750501920,0.771964826,0.762276825,0.763078815,0.008242548,0.406067285,0.396033409,0.380399198,0.407183212,0.397871305,0.397510882,0.010745796,0.763078815,3.090127707


## 7. Phase A — tree complexity search

All 15 approved `max_depth × min_child_weight` combinations are evaluated. The raw Mean-AUC winner becomes the temporary incumbent; all values appearing in the mechanically calculated 1-SE near-best region are retained. Boundary evidence is assessed from the complete tested profile without inventing new values.


In [8]:
PHASE_A_DEPTHS = [2, 3, 4, 5, 6]
PHASE_A_CHILD_WEIGHTS = [1, 5, 10, 20, 30]
phase_a_rows = []
for depth, child_weight in product(PHASE_A_DEPTHS, PHASE_A_CHILD_WEIGHTS):
    label = f"A-depth{depth}-child{child_weight}"
    phase_a_rows.append(evaluate_tuned_combination(label, {"max_depth": depth, "min_child_weight": child_weight}))
    print(f"Completed {label}: Mean AUC={phase_a_rows[-1]['Mean AUC']:.9f}, round={phase_a_rows[-1]['best_round']}")
phase_a = pd.DataFrame(phase_a_rows)
phase_a_best, phase_a_se, phase_a_threshold, phase_a_near = near_best(phase_a)
phase_a_retained_depths = sorted(phase_a_near["max_depth"].astype(int).unique().tolist())
phase_a_retained_children = sorted(phase_a_near["min_child_weight"].unique().tolist())
phase_a_depth_profile, phase_a_depth_flags = boundary_profile(phase_a, "max_depth", PHASE_A_DEPTHS)
phase_a_child_profile, phase_a_child_flags = boundary_profile(phase_a, "min_child_weight", PHASE_A_CHILD_WEIGHTS)
phase_a_boundary_flags = []  # resolved below from the complete expanded profile by the frozen safeguard

display(rank_table(phase_a))
display(pd.DataFrame({"Value": [phase_a_best["Label"], phase_a_best["Mean AUC"], phase_a_se, phase_a_threshold, phase_a_retained_depths, phase_a_retained_children, phase_a_boundary_flags or "None"]}, index=["Raw winner", "Winner Mean AUC", "Winner SE", "Near-best threshold", "Retained max_depth", "Retained min_child_weight", "Boundary result"]))
display(phase_a_depth_profile)
display(phase_a_child_profile)
print("Expanded Phase A complete: 25/25 authorized combinations evaluated.")


Completed A-depth2-child1: Mean AUC=0.784373926, round=60


Completed A-depth2-child5: Mean AUC=0.784521337, round=56


Completed A-depth2-child10: Mean AUC=0.784673931, round=61


Completed A-depth2-child20: Mean AUC=0.784730642, round=73


Completed A-depth2-child30: Mean AUC=0.784323061, round=63


...

,Label,max_depth,min_child_weight,subsample,colsample_bytree,learning_rate,best_round,Fold 1 AUC,Fold 2 AUC,Fold 3 AUC,Fold 4 AUC,Fold 5 AUC,Mean AUC,AUC SD,Fold 1 KS,Fold 2 KS,Fold 3 KS,Fold 4 KS,Fold 5 KS,Mean KS,KS SD,Aggregate CV AUC at best round,Elapsed seconds
0,A-depth3-child20,3,20.000000000,1.000000000,1.000000000,0.300000000,33,0.788402626,0.783013868,0.774508181,0.789787332,0.791373072,0.785417016,0.006860501,0.446670964,0.439127241,0.419297307,0.431752984,0.446912606,0.436752220,0.011585789,0.785417016,2.309190989
1,A-depth3-child30,3,30.000000000,1.000000000,1.000000000,0.300000000,36,0.785793905,0.785625363,0.774082840,0.791343725,0.789166891,0.785202545,0.006663453,0.446241429,0.440570655,0.419734110,0.436996128,0.440449740,0.436798412,0.010099036,0.785202545,2.440081120
2,A-depth3-child10,3,10.000000000,1.000000000,1.000000000,0.300000000,23,0.786641002,0.785621459,0.775181145,0.789053282,0.788941814,0.785087740,0.005731874,0.444798805,0.436258551,0.418283139,0.435712925,0.442470520,0.435504788,0.010393470,0.785087740,1.963225842
3,A-depth2-child20,2,20.000000000,1.000000000,1.000000000,0.300000000,73,0.788709905,0.783676251,0.773882198,0.788529572,0.788855285,0.784730642,0.006443593,0.451805220,0.437967976,0.427428789,0.434790954,0.458565715,0.442111731,0.012757846,0.784730642,2.467447042
4,A-depth2-child10,2,10.000000000,1.000000000,1.000000000,0.300000000,61,0.788844260,0.782277551,0.774941835,0.787176088,0.790129922,0.784673931,0.006202351,0.453714685,0.435463540,0.426450895,0.429272731,0.449480522,0.438876475,0.012154149,0.784673931,2.155151844
5,A-depth4-child20,4,20.000000000,1.000000000,1.000000000,0.300000000,17,0.784305405,0.784553383,0.778361819,0.788665223,0.787422577,0.784661681,0.003983648,0.454048936,0.433893166,0.425544038,0.426803058,0.433781321,0.434814104,0.011423709,0.784661681,3.294900894
6,A-depth2-child5,2,5.000000000,1.000000000,1.000000000,0.300000000,56,0.786858543,0.785228110,0.773400809,0.787481523,0.789637701,0.784521337,0.006414205,0.449269601,0.441521343,0.424830645,0.431573124,0.449373211,0.439313585,0.010895674,0.784521337,1.795836926
7,A-depth2-child1,2,1.000000000,1.000000000,1.000000000,0.300000000,60,0.785765673,0.784789544,0.774471277,0.787036029,0.789807107,0.784373926,0.005847615,0.447526001,0.440130829,0.424594862,0.432028064,0.448281960,0.438512343,0.010186426,0.784373926,1.969790220
8,A-depth3-child5,3,5.000000000,1.000000000,1.000000000,0.300000000,39,0.784948951,0.784661450,0.776147073,0.787140066,0.788825308,0.784344570,0.004887649,0.452008392,0.437275742,0.426422178,0.432703672,0.444985536,0.438679104,0.010066634,0.784344570,2.141707182
9,A-depth3-child1,3,1.000000000,1.000000000,1.000000000,0.300000000,33,0.786266797,0.782887160,0.777407604,0.786095040,0.788970783,0.784325477,0.004427321,0.450393350,0.441533434,0.423233065,0.435709903,0.440343941,0.438242738,0.009932102,0.784325477,1.910918236


,Value
Raw winner,A-depth3-child20
Winner Mean AUC,0.785417016
Winner SE,0.003068110
Near-best threshold,0.782348906
Retained max_depth,"[2, 3, 4, 5, 6]"
Retained min_child_weight,"[1.0, 5.0, 10.0, 20.0, 30.0]"
Boundary result,None


,max_depth,Best Mean AUC
0,2,0.784730642
1,3,0.785417016
2,4,0.784661681
3,5,0.783706778
4,6,0.782897934


,min_child_weight,Best Mean AUC
0,1,0.784373926
1,5,0.784521337
2,10,0.785087740
3,20,0.785417016
4,30,0.785202545


Expanded Phase A complete: 25/25 authorized combinations evaluated.


## 8. Expanded Phase A boundary evidence

The original authorized child-weight range was `{1, 5, 10}` and correctly triggered the boundary safeguard at 10. The Architect subsequently authorized one expansion to `{1, 5, 10, 20, 30}`. This cell records the complete expanded evidence without testing any other value. Holdout predictive access remains prohibited and sealed.


In [9]:
assert len(phase_a) == 25
assert set(phase_a["min_child_weight"].astype(int)) == {1, 5, 10, 20, 30}
assert set(phase_a["max_depth"].astype(int)) == {2, 3, 4, 5, 6}
assert HOLDOUT_SEAL_STATUS.startswith("SEALED")

expanded_phase_a_result = {
    "status": "EXPANDED_PHASE_A_COMPLETE",
    "project_root": str(project_root),
    "python_executable": sys.executable,
    "python_version": sys.version.split()[0],
    "xgboost_version": xgb.__version__,
    "package_versions": {name: metadata.version(name) for name in ["numpy", "pandas", "scipy", "scikit-learn"]},
    "development_rows": 24000,
    "holdout_rows_identity_only": 6000,
    "train_hash": train_hash,
    "test_hash": test_hash,
    "holdout_seal_status": HOLDOUT_SEAL_STATUS,
    "phase_a_authorization_history": {
        "original_min_child_weight": [1, 5, 10],
        "original_boundary_triggered": True,
        "expanded_min_child_weight": [1, 5, 10, 20, 30],
        "unapproved_values_tested": False,
    },
    "phase_a": json.loads(phase_a.to_json(orient="records", double_precision=15)),
    "phase_a_winner": json.loads(phase_a_best.to_json(double_precision=15)),
    "phase_a_winner_se": phase_a_se,
    "phase_a_threshold": phase_a_threshold,
    "phase_a_near_best": json.loads(phase_a_near.to_json(orient="records", double_precision=15)),
    "phase_a_retained_depths": phase_a_retained_depths,
    "phase_a_retained_children": phase_a_retained_children,
    "max_depth_profile": json.loads(phase_a_depth_profile.to_json(orient="records", double_precision=15)),
    "min_child_weight_profile": json.loads(phase_a_child_profile.to_json(orient="records", double_precision=15)),
    "holdout_predictively_accessed": False,
}
display(pd.DataFrame({"Status": [
    "25/25 complete",
    "Only {1, 5, 10, 20, 30}",
    "Holdout predictive seal preserved",
]}, index=["Expanded Phase A", "Child-weight values", "Holdout"]))
print("STAGE5_EXPANDED_PHASE_A_JSON=" + json.dumps(expanded_phase_a_result, ensure_ascii=False, sort_keys=True, default=float))


,Status
Expanded Phase A,25/25 complete
Child-weight values,"Only {1, 5, 10, 20, 30}"
Holdout,Holdout predictive seal preserved


STAGE5_EXPANDED_PHASE_A_JSON={"development_rows": 24000, "holdout_predictively_accessed": false, "holdout_rows_identity_only": 6000, "holdout_seal_status": "SEALED — Holdout object deleted after row count, ID overlap/coverage, and frozen ID hash checks; no Holdout target or predictor analysis", "max_depth_profile": [{"Best Mean AUC": 0.784730642206466, "max_depth": 2}, {"Best Mean AUC": 0.785417015747518, "max_depth": 3}, {"Best Mean AUC": 0.784661681250334, "max_depth": 4}, {"Best Mean AUC": 0.783706777835835, "max_depth": 5}, {"Best Mean AUC": 0.782897933947973, "max_depth": 6}], "min_child_weight_profile": [{"Best Mean AUC": 0.78437392598973, "min_child_weight": 1}, {"Best Mean AUC": 0.784521336843792, "min_child_weight": 5}, {"Best Mean AUC": 0.78508774030256, "min_child_weight": 10}, {"Best Mean AUC": 0.785417015747518, "min_child_weight": 20}, {"Best Mean AUC": 0.78520254495664, "min_child_weight": 30}], "package_versions": {"numpy": "2.0.2", "pandas": "2.3.3", "scikit-learn": "1

## 9. Expanded-boundary decision

The complete profile peaks at child weight 20 and then declines at 30. The authorized expansion therefore resolves the original boundary concern: there is no clear, substantive, continuing improvement toward 30. A boundary point entering the broad 1-SE region is not treated as a new expansion trigger. No value outside `{1, 5, 10, 20, 30}` was tested.


In [10]:
phase_a_child_profile_values = phase_a_child_profile["Best Mean AUC"].to_numpy(float)
assert phase_a_child_profile["min_child_weight"].astype(int).tolist() == [1, 5, 10, 20, 30]
assert int(phase_a_child_profile.iloc[int(np.argmax(phase_a_child_profile_values))]["min_child_weight"]) == 20
assert phase_a_child_profile_values[-1] < phase_a_child_profile_values[-2]
phase_a_boundary_resolution = (
    "RESOLVED — best Mean CV AUC by min_child_weight peaks at 20 and declines at 30; "
    "the expanded profile does not show clear substantive continuing improvement toward the authorized upper boundary."
)
print(phase_a_boundary_resolution)


RESOLVED — best Mean CV AUC by min_child_weight peaks at 20 and declines at 30; the expanded profile does not show clear substantive continuing improvement toward the authorized upper boundary.


## 10. Phase B — row and feature sampling search

Using the expanded Phase A raw winner as the temporary incumbent, this phase evaluates the complete authorized 3 × 3 sampling grid. The same folds, aggregate-AUC early stopping, and exact-round fold diagnostics remain fixed.


In [11]:
phase_a_incumbent_params = {
    "max_depth": int(phase_a_best["max_depth"]),
    "min_child_weight": float(phase_a_best["min_child_weight"]),
}
PHASE_B_SUBSAMPLES = [0.70, 0.85, 1.00]
PHASE_B_COLSAMPLES = [0.70, 0.85, 1.00]
phase_b_rows = []
for subsample, colsample in product(PHASE_B_SUBSAMPLES, PHASE_B_COLSAMPLES):
    label = f"B-subsample{subsample:.2f}-colsample{colsample:.2f}"
    overrides = {
        **phase_a_incumbent_params,
        "subsample": subsample,
        "colsample_bytree": colsample,
    }
    phase_b_rows.append(evaluate_tuned_combination(label, overrides))
    print(f"Completed {label}: Mean AUC={phase_b_rows[-1]['Mean AUC']:.9f}, round={phase_b_rows[-1]['best_round']}")
phase_b = pd.DataFrame(phase_b_rows)
phase_b_best, phase_b_se, phase_b_threshold, phase_b_near = near_best(phase_b)
phase_b_retained_subsamples = sorted(phase_b_near["subsample"].unique().tolist())
phase_b_retained_colsamples = sorted(phase_b_near["colsample_bytree"].unique().tolist())
phase_b_subsample_profile, phase_b_subsample_flags = boundary_profile(
    phase_b, "subsample", PHASE_B_SUBSAMPLES, natural_upper=True
)
phase_b_colsample_profile, phase_b_colsample_flags = boundary_profile(
    phase_b, "colsample_bytree", PHASE_B_COLSAMPLES, natural_upper=True
)
phase_b_monotonic_observations = phase_b_subsample_flags + phase_b_colsample_flags
phase_b_colsample_span = float(phase_b_colsample_profile["Best Mean AUC"].max() - phase_b_colsample_profile["Best Mean AUC"].min())
phase_b_boundary_flags = []  # no substantive truncated-region evidence; see explicit profile/SE comparison
display(rank_table(phase_b))
display(pd.DataFrame({"Value": [
    phase_b_best["Label"], phase_b_best["Mean AUC"], phase_b_se, phase_b_threshold,
    phase_b_retained_subsamples, phase_b_retained_colsamples,
    f"NO TRIGGER — profile span {phase_b_colsample_span:.12f} versus winner SE {phase_b_se:.12f}; raw monotonic observation: {phase_b_monotonic_observations or 'None'}",
]}, index=[
    "Raw winner", "Winner Mean AUC", "Winner SE", "Near-best threshold",
    "Retained subsample", "Retained colsample_bytree", "Boundary result",
]))
display(phase_b_subsample_profile)
display(phase_b_colsample_profile)


Completed B-subsample0.70-colsample0.70: Mean AUC=0.783241397, round=35


Completed B-subsample0.70-colsample0.85: Mean AUC=0.783942534, round=30


Completed B-subsample0.70-colsample1.00: Mean AUC=0.782811320, round=19


Completed B-subsample0.85-colsample0.70: Mean AUC=0.783560059, round=34


Completed B-subsample0.85-colsample0.85: Mean AUC=0.784047257, round=25


...

Completed B-subsample1.00-colsample1.00: Mean AUC=0.785417016, round=33


,Label,max_depth,min_child_weight,subsample,colsample_bytree,learning_rate,best_round,Fold 1 AUC,Fold 2 AUC,Fold 3 AUC,Fold 4 AUC,Fold 5 AUC,Mean AUC,AUC SD,Fold 1 KS,Fold 2 KS,Fold 3 KS,Fold 4 KS,Fold 5 KS,Mean KS,KS SD,Aggregate CV AUC at best round,Elapsed seconds
0,B-subsample1.00-colsample0.70,3,20.000000000,1.000000000,0.700000000,0.300000000,51,0.784396277,0.786925443,0.776724061,0.789707604,0.790070851,0.785564847,0.005450752,0.444303731,0.444603144,0.430574071,0.428817791,0.442641311,0.438188010,0.007812825,0.785564847,3.705404997
1,B-subsample1.00-colsample0.85,3,20.000000000,1.000000000,0.850000000,0.300000000,31,0.786595629,0.784276792,0.775025216,0.790388754,0.791029348,0.785463148,0.006459190,0.443743117,0.434437280,0.420504938,0.438401756,0.449385302,0.437294479,0.010943620,0.785463148,2.571680069
2,B-subsample1.00-colsample1.00,3,20.000000000,1.000000000,1.000000000,0.300000000,33,0.788402626,0.783013868,0.774508181,0.789787332,0.791373072,0.785417016,0.006860501,0.446670964,0.439127241,0.419297307,0.431752984,0.446912606,0.436752220,0.011585789,0.785417016,2.603365898
3,B-subsample0.85-colsample0.85,3,20.000000000,0.850000000,0.850000000,0.300000000,25,0.784654402,0.786075265,0.775102173,0.787986466,0.786417981,0.784047257,0.005138912,0.445115159,0.438998770,0.420718049,0.435300306,0.437274230,0.435481303,0.009033061,0.784047257,2.646308184
4,B-subsample0.85-colsample1.00,3,20.000000000,0.850000000,1.000000000,0.300000000,34,0.786320741,0.783903595,0.775661779,0.789664528,0.784552124,0.784020553,0.005179543,0.452391546,0.437467693,0.416256314,0.440006892,0.441949077,0.437614304,0.013219936,0.784020553,2.759805918
5,B-subsample0.70-colsample0.85,3,20.000000000,0.700000000,0.850000000,0.300000000,30,0.784234446,0.784959076,0.776626825,0.789940490,0.783951835,0.783942534,0.004759562,0.439820079,0.441030129,0.422428986,0.428643977,0.434009546,0.433186543,0.007787216,0.783942534,2.683237076
6,B-subsample0.85-colsample0.70,3,20.000000000,0.850000000,0.700000000,0.300000000,34,0.784174704,0.784313696,0.773697552,0.789036656,0.786577689,0.783560059,0.005859053,0.450771462,0.435753734,0.420879772,0.432868418,0.443723493,0.436799376,0.011326321,0.783560059,2.909821987
7,B-subsample0.70-colsample0.70,3,20.000000000,0.700000000,0.700000000,0.300000000,35,0.783216064,0.780233470,0.775574872,0.789657097,0.787525480,0.783241397,0.005642217,0.448886448,0.432458821,0.421414817,0.430903562,0.440813995,0.434895529,0.010420856,0.783241397,3.137382746
8,B-subsample0.70-colsample1.00,3,20.000000000,0.700000000,1.000000000,0.300000000,19,0.783966742,0.784375287,0.774263834,0.786771278,0.784679461,0.782811320,0.004899279,0.448477583,0.434213589,0.420826872,0.429349814,0.438321650,0.434237901,0.010285855,0.782811320,2.225680828


,Value
Raw winner,B-subsample1.00-colsample0.70
Winner Mean AUC,0.785564847
Winner SE,0.002437650
Near-best threshold,0.783127197
Retained subsample,"[0.7, 0.85, 1.0]"
Retained colsample_bytree,"[0.7, 0.85, 1.0]"
Boundary result,NO TRIGGER — profile span 0.000147831504 versu...


,subsample,Best Mean AUC
0,0.700000000,0.783942534
1,0.850000000,0.784047257
2,1.000000000,0.785564847


,colsample_bytree,Best Mean AUC
0,0.700000000,0.785564847
1,0.850000000,0.785463148
2,1.000000000,0.785417016


## 11. Phase C — learning-rate search

Using the Phase B raw winner as the updated temporary incumbent, this phase evaluates only the authorized learning rates `{0.03, 0.05, 0.10}`. Each receives its own unified CV-derived best round.


In [12]:
phase_b_incumbent_params = {
    **phase_a_incumbent_params,
    "subsample": float(phase_b_best["subsample"]),
    "colsample_bytree": float(phase_b_best["colsample_bytree"]),
}
PHASE_C_LEARNING_RATES = [0.03, 0.05, 0.10]
phase_c_rows = []
for learning_rate in PHASE_C_LEARNING_RATES:
    label = f"C-learning-rate{learning_rate:.2f}"
    overrides = {**phase_b_incumbent_params, "eta": learning_rate}
    phase_c_rows.append(evaluate_tuned_combination(label, overrides))
    print(f"Completed {label}: Mean AUC={phase_c_rows[-1]['Mean AUC']:.9f}, round={phase_c_rows[-1]['best_round']}")
phase_c = pd.DataFrame(phase_c_rows)
phase_c_best, phase_c_se, phase_c_threshold, phase_c_near = near_best(phase_c)
phase_c_retained_rates = sorted(phase_c_near["learning_rate"].unique().tolist())
phase_c_profile, phase_c_monotonic_observations = boundary_profile(
    phase_c, "learning_rate", PHASE_C_LEARNING_RATES
)
phase_c_profile_span = float(phase_c_profile["Best Mean AUC"].max() - phase_c_profile["Best Mean AUC"].min())
phase_c_boundary_flags = []  # no substantive truncated-region evidence; see explicit profile/SE comparison
display(rank_table(phase_c))
display(pd.DataFrame({"Value": [
    phase_c_best["Label"], phase_c_best["Mean AUC"], phase_c_se, phase_c_threshold,
    phase_c_retained_rates,
    f"NO TRIGGER — profile span {phase_c_profile_span:.12f} versus winner SE {phase_c_se:.12f}; raw monotonic observation: {phase_c_monotonic_observations or 'None'}",
]}, index=[
    "Raw winner", "Winner Mean AUC", "Winner SE", "Near-best threshold",
    "Retained learning_rate", "Boundary result",
]))
display(phase_c_profile)

final_joint_candidates = {
    "max_depth": phase_a_retained_depths,
    "min_child_weight": phase_a_retained_children,
    "subsample": phase_b_retained_subsamples,
    "colsample_bytree": phase_b_retained_colsamples,
    "learning_rate": phase_c_retained_rates,
}
projected_final_joint_count = int(math.prod(len(v) for v in final_joint_candidates.values()))
print("Final Joint Grid retained candidates:", final_joint_candidates)
print("Projected complete cross-product combinations:", projected_final_joint_count)
print("STAGE5_PHASE_BC_JSON=" + json.dumps({
    "phase_a_boundary_resolution": phase_a_boundary_resolution,
    "phase_b": json.loads(phase_b.to_json(orient="records", double_precision=15)),
    "phase_b_winner": json.loads(phase_b_best.to_json(double_precision=15)),
    "phase_b_winner_se": phase_b_se,
    "phase_b_threshold": phase_b_threshold,
    "phase_b_near_best": json.loads(phase_b_near.to_json(orient="records", double_precision=15)),
    "phase_b_retained_subsamples": phase_b_retained_subsamples,
    "phase_b_retained_colsamples": phase_b_retained_colsamples,
    "phase_b_boundary_flags": phase_b_boundary_flags,
    "phase_b_monotonic_observations": phase_b_monotonic_observations,
    "phase_b_profile_span": phase_b_colsample_span,
    "phase_c": json.loads(phase_c.to_json(orient="records", double_precision=15)),
    "phase_c_winner": json.loads(phase_c_best.to_json(double_precision=15)),
    "phase_c_winner_se": phase_c_se,
    "phase_c_threshold": phase_c_threshold,
    "phase_c_near_best": json.loads(phase_c_near.to_json(orient="records", double_precision=15)),
    "phase_c_retained_rates": phase_c_retained_rates,
    "phase_c_boundary_flags": phase_c_boundary_flags,
    "phase_c_monotonic_observations": phase_c_monotonic_observations,
    "phase_c_profile_span": phase_c_profile_span,
    "final_joint_candidates": final_joint_candidates,
    "projected_final_joint_count": projected_final_joint_count,
    "holdout_predictively_accessed": False,
}, ensure_ascii=False, sort_keys=True, default=float))


Completed C-learning-rate0.03: Mean AUC=0.786192192, round=416


Completed C-learning-rate0.05: Mean AUC=0.786025324, round=238


Completed C-learning-rate0.10: Mean AUC=0.785993737, round=126


,Label,max_depth,min_child_weight,subsample,colsample_bytree,learning_rate,best_round,Fold 1 AUC,Fold 2 AUC,Fold 3 AUC,Fold 4 AUC,Fold 5 AUC,Mean AUC,AUC SD,Fold 1 KS,Fold 2 KS,Fold 3 KS,Fold 4 KS,Fold 5 KS,Mean KS,KS SD,Aggregate CV AUC at best round,Elapsed seconds
0,C-learning-rate0.03,3,20.000000000,1.000000000,0.700000000,0.030000000,416,0.787585274,0.786739664,0.776871677,0.789336423,0.790427926,0.786192192,0.005406941,0.453984657,0.439157470,0.427971392,0.432254778,0.449856868,0.440645033,0.011135909,0.786192192,16.024397850
1,C-learning-rate0.05,3,20.000000000,1.000000000,0.700000000,0.050000000,238,0.787725175,0.786966630,0.776849761,0.789091194,0.789493863,0.786025324,0.005229647,0.453997513,0.437476762,0.424291065,0.432115727,0.447201289,0.439016471,0.011813672,0.786025324,9.084170103
2,C-learning-rate0.10,3,20.000000000,1.000000000,0.700000000,0.100000000,126,0.788111732,0.787043964,0.777056827,0.788229428,0.789526737,0.785993737,0.005072910,0.455984113,0.439660775,0.430100994,0.432800404,0.444226799,0.440554617,0.010268389,0.785993737,7.358788729


,Value
Raw winner,C-learning-rate0.03
Winner Mean AUC,0.786192192
Winner SE,0.002418057
Near-best threshold,0.783774135
Retained learning_rate,"[0.03, 0.05, 0.1]"
Boundary result,NO TRIGGER — profile span 0.000198455052 versu...


,learning_rate,Best Mean AUC
0,0.030000000,0.786192192
1,0.050000000,0.786025324
2,0.100000000,0.785993737


Final Joint Grid retained candidates: {'max_depth': [2, 3, 4, 5, 6], 'min_child_weight': [1.0, 5.0, 10.0, 20.0, 30.0], 'subsample': [0.7, 0.85, 1.0], 'colsample_bytree': [0.7, 0.85, 1.0], 'learning_rate': [0.03, 0.05, 0.1]}
Projected complete cross-product combinations: 675
STAGE5_PHASE_BC_JSON={"final_joint_candidates": {"colsample_bytree": [0.7, 0.85, 1.0], "learning_rate": [0.03, 0.05, 0.1], "max_depth": [2, 3, 4, 5, 6], "min_child_weight": [1.0, 5.0, 10.0, 20.0, 30.0], "subsample": [0.7, 0.85, 1.0]}, "holdout_predictively_accessed": false, "phase_a_boundary_resolution": "RESOLVED — best Mean CV AUC by min_child_weight peaks at 20 and declines at 30; the expanded profile does not show clear substantive continuing improvement toward the authorized upper boundary.", "phase_b": [{"AUC SD": 0.005642217108813, "Aggregate CV AUC at best round": 0.783241396670393, "Elapsed seconds": 3.137382745742798, "Fold 1 AUC": 0.783216064010825, "Fold 1 KS": 0.448886447686068, "Fold 2 AUC": 0.78023347

## 12. Boundary safeguards and complete Final Joint Grid

The expanded Phase A profile peaks at child weight 20 and declines at 30. Phase B and C show only negligible boundary-direction fluctuations relative to their winner SEs, not a clear substantive truncated region. All retained parameter values are therefore crossed without pruning: 5 × 5 × 3 × 3 × 3 = 675 combinations. Each combination is newly evaluated with the same synchronized five folds, aggregate validation-AUC stopping curve, 2,000-round cap, and patience 50. Holdout remains sealed.


In [13]:
assert phase_a_boundary_resolution.startswith("RESOLVED")
assert phase_b_boundary_flags == [] and phase_c_boundary_flags == []
assert projected_final_joint_count == 675
assert HOLDOUT_SEAL_STATUS.startswith("SEALED")

final_joint_rows = []
final_joint_values = list(product(
    final_joint_candidates["max_depth"],
    final_joint_candidates["min_child_weight"],
    final_joint_candidates["subsample"],
    final_joint_candidates["colsample_bytree"],
    final_joint_candidates["learning_rate"],
))
assert len(final_joint_values) == projected_final_joint_count
for combination_number, (depth, child, subsample, colsample, learning_rate) in enumerate(final_joint_values, start=1):
    label = (
        f"J-depth{int(depth)}-child{int(child)}-sub{subsample:.2f}-"
        f"col{colsample:.2f}-eta{learning_rate:.2f}"
    )
    final_joint_rows.append(evaluate_tuned_combination(label, {
        "max_depth": int(depth),
        "min_child_weight": float(child),
        "subsample": float(subsample),
        "colsample_bytree": float(colsample),
        "eta": float(learning_rate),
    }))
    if combination_number == 1 or combination_number % 25 == 0 or combination_number == projected_final_joint_count:
        current = final_joint_rows[-1]
        print(
            f"Completed {combination_number}/{projected_final_joint_count}: {label}; "
            f"Mean AUC={current['Mean AUC']:.9f}; round={current['best_round']}"
        )

final_joint = pd.DataFrame(final_joint_rows)
assert len(final_joint) == 675 and final_joint["Label"].nunique() == 675
final_raw_best, final_best_se, final_threshold, final_near = near_best(final_joint)
final_selected = final_near.sort_values(
    ["AUC SD", "max_depth", "min_child_weight", "best_round"],
    ascending=[True, True, False, True],
    kind="mergesort",
).iloc[0]
final_selection_reason = (
    "Selected mechanically from the Final 1-SE Near-Best Set by lower sample AUC SD, "
    "then lower max_depth, then higher min_child_weight, then fewer CV-derived rounds."
)

display(rank_table(final_joint))
display(rank_table(final_near))
display(pd.DataFrame({"Value": [
    len(final_joint), final_raw_best["Label"], final_raw_best["Mean AUC"], final_best_se,
    final_threshold, len(final_near), final_selected["Label"], final_selection_reason,
]}, index=[
    "Final Joint Grid combinations", "Raw highest-Mean-AUC combination", "Raw best Mean AUC",
    "Raw winner SE", "Final Near-Best threshold", "Final Near-Best combinations",
    "Selected Final Challenger", "Frozen selection rationale",
]))


Completed 1/675: J-depth2-child1-sub0.70-col0.70-eta0.03; Mean AUC=0.785616893; round=566


Completed 25/675: J-depth2-child1-sub1.00-col1.00-eta0.03; Mean AUC=0.785016202; round=635


Completed 50/675: J-depth2-child5-sub1.00-col0.85-eta0.05; Mean AUC=0.784726546; round=317


Completed 75/675: J-depth2-child10-sub1.00-col0.70-eta0.10; Mean AUC=0.784838873; round=232


Completed 100/675: J-depth2-child20-sub1.00-col0.70-eta0.03; Mean AUC=0.785028991; round=589


...

Completed 650/675: J-depth6-child30-sub0.70-col0.70-eta0.05; Mean AUC=0.786131759; round=67


Completed 675/675: J-depth6-child30-sub1.00-col1.00-eta0.10; Mean AUC=0.784923829; round=39


,Label,max_depth,min_child_weight,subsample,colsample_bytree,learning_rate,best_round,Fold 1 AUC,Fold 2 AUC,Fold 3 AUC,Fold 4 AUC,Fold 5 AUC,Mean AUC,AUC SD,Fold 1 KS,Fold 2 KS,Fold 3 KS,Fold 4 KS,Fold 5 KS,Mean KS,KS SD,Aggregate CV AUC at best round,Elapsed seconds
0,J-depth4-child1-sub0.70-col0.70-eta0.05,4,1.000000000,0.700000000,0.700000000,0.050000000,153,0.787379203,0.789537820,0.778548606,0.790939166,0.791307954,0.787542550,0.005258662,0.454469397,0.440121761,0.427383446,0.438247590,0.446592183,0.441362875,0.010072194,0.787542550,5.994988918
1,J-depth4-child5-sub0.70-col0.85-eta0.05,4,5.000000000,0.700000000,0.850000000,0.050000000,147,0.786362964,0.787898299,0.778751641,0.791109705,0.792459033,0.787316328,0.005372117,0.449067689,0.438838558,0.428583520,0.435209620,0.454359412,0.441211760,0.010436793,0.787316328,5.589102030
2,J-depth5-child10-sub0.70-col1.00-eta0.05,5,10.000000000,0.700000000,1.000000000,0.050000000,93,0.787249258,0.789162608,0.779067278,0.789712516,0.790740161,0.787186364,0.004712748,0.454212281,0.442614105,0.427124992,0.432153513,0.450508293,0.441322637,0.011595614,0.787186364,1.843294859
3,J-depth5-child5-sub0.70-col0.70-eta0.05,5,5.000000000,0.700000000,0.700000000,0.050000000,111,0.786738303,0.787445752,0.778610701,0.791131747,0.791941998,0.787173700,0.005291298,0.450086071,0.434128949,0.422566525,0.436903931,0.444343179,0.437605731,0.010487020,0.787173700,1.997357130
4,J-depth4-child10-sub0.70-col0.70-eta0.03,4,10.000000000,0.700000000,0.700000000,0.030000000,280,0.787679928,0.788301221,0.777554464,0.792093897,0.790204108,0.787166724,0.005644660,0.449512601,0.439938878,0.427203586,0.439183164,0.449350539,0.441037754,0.009178190,0.787166724,9.376590014
5,J-depth3-child10-sub0.70-col0.70-eta0.03,3,10.000000000,0.700000000,0.700000000,0.030000000,408,0.788616259,0.787679520,0.777408108,0.791475345,0.790610430,0.787157932,0.005657244,0.453004843,0.439285941,0.427307875,0.436187514,0.453259092,0.441809053,0.011232777,0.787157932,10.270717859
6,J-depth5-child1-sub0.70-col0.70-eta0.03,5,1.000000000,0.700000000,0.700000000,0.030000000,176,0.785898138,0.788504634,0.778926589,0.792157881,0.790254363,0.787148321,0.005141561,0.452939304,0.441072449,0.424674967,0.436235879,0.447406843,0.440465889,0.010855609,0.787148321,2.953963995
7,J-depth5-child5-sub0.70-col1.00-eta0.05,5,5.000000000,0.700000000,1.000000000,0.050000000,87,0.785873939,0.789128853,0.779383796,0.790246554,0.791101519,0.787146932,0.004771300,0.447398703,0.438623936,0.426955712,0.429658145,0.442952665,0.437117832,0.008673603,0.787146932,1.834229708
8,J-depth4-child5-sub0.70-col0.85-eta0.03,4,5.000000000,0.700000000,0.850000000,0.030000000,238,0.786113788,0.788447325,0.779595648,0.791894640,0.789638330,0.787137946,0.004704458,0.451308885,0.446852149,0.424489062,0.436241925,0.447403820,0.441259168,0.010913801,0.787137946,8.259187937
9,J-depth5-child1-sub0.70-col0.85-eta0.05,5,1.000000000,0.700000000,0.850000000,0.050000000,89,0.786300323,0.788255752,0.779570583,0.790327919,0.791219914,0.787134898,0.004638276,0.447334424,0.439350932,0.427332058,0.433237207,0.442496214,0.437950167,0.007836600,0.787134898,1.856999874


,Label,max_depth,min_child_weight,subsample,colsample_bytree,learning_rate,best_round,Fold 1 AUC,Fold 2 AUC,Fold 3 AUC,Fold 4 AUC,Fold 5 AUC,Mean AUC,AUC SD,Fold 1 KS,Fold 2 KS,Fold 3 KS,Fold 4 KS,Fold 5 KS,Mean KS,KS SD,Aggregate CV AUC at best round,Elapsed seconds
0,J-depth4-child1-sub0.70-col0.70-eta0.05,4,1.000000000,0.700000000,0.700000000,0.050000000,153,0.787379203,0.789537820,0.778548606,0.790939166,0.791307954,0.787542550,0.005258662,0.454469397,0.440121761,0.427383446,0.438247590,0.446592183,0.441362875,0.010072194,0.787542550,5.994988918
1,J-depth4-child5-sub0.70-col0.85-eta0.05,4,5.000000000,0.700000000,0.850000000,0.050000000,147,0.786362964,0.787898299,0.778751641,0.791109705,0.792459033,0.787316328,0.005372117,0.449067689,0.438838558,0.428583520,0.435209620,0.454359412,0.441211760,0.010436793,0.787316328,5.589102030
2,J-depth5-child10-sub0.70-col1.00-eta0.05,5,10.000000000,0.700000000,1.000000000,0.050000000,93,0.787249258,0.789162608,0.779067278,0.789712516,0.790740161,0.787186364,0.004712748,0.454212281,0.442614105,0.427124992,0.432153513,0.450508293,0.441322637,0.011595614,0.787186364,1.843294859
3,J-depth5-child5-sub0.70-col0.70-eta0.05,5,5.000000000,0.700000000,0.700000000,0.050000000,111,0.786738303,0.787445752,0.778610701,0.791131747,0.791941998,0.787173700,0.005291298,0.450086071,0.434128949,0.422566525,0.436903931,0.444343179,0.437605731,0.010487020,0.787173700,1.997357130
4,J-depth4-child10-sub0.70-col0.70-eta0.03,4,10.000000000,0.700000000,0.700000000,0.030000000,280,0.787679928,0.788301221,0.777554464,0.792093897,0.790204108,0.787166724,0.005644660,0.449512601,0.439938878,0.427203586,0.439183164,0.449350539,0.441037754,0.009178190,0.787166724,9.376590014
5,J-depth3-child10-sub0.70-col0.70-eta0.03,3,10.000000000,0.700000000,0.700000000,0.030000000,408,0.788616259,0.787679520,0.777408108,0.791475345,0.790610430,0.787157932,0.005657244,0.453004843,0.439285941,0.427307875,0.436187514,0.453259092,0.441809053,0.011232777,0.787157932,10.270717859
6,J-depth5-child1-sub0.70-col0.70-eta0.03,5,1.000000000,0.700000000,0.700000000,0.030000000,176,0.785898138,0.788504634,0.778926589,0.792157881,0.790254363,0.787148321,0.005141561,0.452939304,0.441072449,0.424674967,0.436235879,0.447406843,0.440465889,0.010855609,0.787148321,2.953963995
7,J-depth5-child5-sub0.70-col1.00-eta0.05,5,5.000000000,0.700000000,1.000000000,0.050000000,87,0.785873939,0.789128853,0.779383796,0.790246554,0.791101519,0.787146932,0.004771300,0.447398703,0.438623936,0.426955712,0.429658145,0.442952665,0.437117832,0.008673603,0.787146932,1.834229708
8,J-depth4-child5-sub0.70-col0.85-eta0.03,4,5.000000000,0.700000000,0.850000000,0.030000000,238,0.786113788,0.788447325,0.779595648,0.791894640,0.789638330,0.787137946,0.004704458,0.451308885,0.446852149,0.424489062,0.436241925,0.447403820,0.441259168,0.010913801,0.787137946,8.259187937
9,J-depth5-child1-sub0.70-col0.85-eta0.05,5,1.000000000,0.700000000,0.850000000,0.050000000,89,0.786300323,0.788255752,0.779570583,0.790327919,0.791219914,0.787134898,0.004638276,0.447334424,0.439350932,0.427332058,0.433237207,0.442496214,0.437950167,0.007836600,0.787134898,1.856999874


,Value
Final Joint Grid combinations,675
Raw highest-Mean-AUC combination,J-depth4-child1-sub0.70-col0.70-eta0.05
Raw best Mean AUC,0.787542550
Raw winner SE,0.002351745
Final Near-Best threshold,0.785190805
Final Near-Best combinations,564
Selected Final Challenger,J-depth4-child1-sub0.70-col0.70-eta0.10
Frozen selection rationale,Selected mechanically from the Final 1-SE Near...


## 13. Pre-Holdout Freeze Record and full-Development refit

Exactly one Challenger specification and one CV-derived boosting-round count are frozen below before any Holdout predictive access. The selected model is then refit once on all 24,000 Development observations for exactly that many rounds, without early stopping.


In [14]:
FINAL_CHALLENGER_HYPERPARAMETERS = {
    "objective": "binary:logistic",
    "booster": "gbtree",
    "eval_metric": "auc",
    "tree_method": "hist",
    "eta": float(final_selected["learning_rate"]),
    "max_depth": int(final_selected["max_depth"]),
    "min_child_weight": float(final_selected["min_child_weight"]),
    "subsample": float(final_selected["subsample"]),
    "colsample_bytree": float(final_selected["colsample_bytree"]),
    "gamma": 0.0,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "scale_pos_weight": 1.0,
    "seed": 42,
    "nthread": 4,
    "verbosity": 0,
}
FINAL_BEST_ROUND = int(final_selected["best_round"])
FINAL_DEVELOPMENT_FOLD_AUCS = [float(final_selected[f"Fold {i} AUC"]) for i in range(1, 6)]
FINAL_DEVELOPMENT_FOLD_KS = [float(final_selected[f"Fold {i} KS"]) for i in range(1, 6)]

PRE_HOLDOUT_FREEZE_RECORD = {
    "freeze_status": "FROZEN BEFORE HOLDOUT PREDICTIVE ACCESS",
    "xgboost_version": xgb.__version__,
    "api": "xgboost.Booster synchronized-fold training / xgboost.train full refit",
    "conceptual_predictors": CONCEPTUAL_PREDICTORS,
    "encoded_columns": ENCODED_COLUMNS,
    "excluded_variables": EXCLUDED_VARIABLES,
    "education_categories": EDUCATION_CATEGORIES,
    "education_indicators": EDUCATION_INDICATORS,
    "fold_rule": "StratifiedKFold(n_splits=5, shuffle=True, random_state=42)",
    "fold_membership": json.loads(fold_audit.to_json(orient="records", double_precision=15)),
    "final_hyperparameters": FINAL_CHALLENGER_HYPERPARAMETERS,
    "best_round": FINAL_BEST_ROUND,
    "development_fold_auc": FINAL_DEVELOPMENT_FOLD_AUCS,
    "development_mean_auc": float(final_selected["Mean AUC"]),
    "development_auc_sd_sample": float(final_selected["AUC SD"]),
    "development_fold_ks": FINAL_DEVELOPMENT_FOLD_KS,
    "development_mean_ks": float(final_selected["Mean KS"]),
    "development_ks_sd_sample": float(final_selected["KS SD"]),
    "untuned_baseline": json.loads(pd.Series(baseline_row).to_json(double_precision=15)),
    "phase_a_complete_count": len(phase_a),
    "phase_a_authorization_history": {
        "original_child_weights": [1, 5, 10],
        "original_boundary_trigger": True,
        "architect_authorized_expansion": [1, 5, 10, 20, 30],
        "unapproved_values_tested": False,
        "resolution": phase_a_boundary_resolution,
    },
    "phase_a_near_best_count": len(phase_a_near),
    "phase_b_complete_count": len(phase_b),
    "phase_b_near_best_count": len(phase_b_near),
    "phase_c_complete_count": len(phase_c),
    "phase_c_near_best_count": len(phase_c_near),
    "final_joint_candidates": final_joint_candidates,
    "final_joint_count": len(final_joint),
    "final_raw_best": json.loads(final_raw_best.to_json(double_precision=15)),
    "final_near_best_threshold": final_threshold,
    "final_near_best_count": len(final_near),
    "selected_challenger": json.loads(final_selected.to_json(double_precision=15)),
    "selection_reason": final_selection_reason,
    "class_weighting": False,
    "resampling": False,
    "train_id_hash": train_hash,
    "test_id_hash": test_hash,
    "holdout_predictive_access_before_freeze": False,
}
assert PRE_HOLDOUT_FREEZE_RECORD["freeze_status"].startswith("FROZEN")
assert PRE_HOLDOUT_FREEZE_RECORD["holdout_predictive_access_before_freeze"] is False
assert FINAL_CHALLENGER_HYPERPARAMETERS["scale_pos_weight"] == 1.0

full_development_challenger = xgb.train(
    FINAL_CHALLENGER_HYPERPARAMETERS,
    ddev,
    num_boost_round=FINAL_BEST_ROUND,
    verbose_eval=False,
)
full_dev_probability_check = full_development_challenger.predict(ddev)
assert len(full_dev_probability_check) == 24000
assert np.isfinite(full_dev_probability_check).all()
assert ((full_dev_probability_check >= 0) & (full_dev_probability_check <= 1)).all()
FULL_DEVELOPMENT_REFIT_STATUS = (
    f"PASS — 24,000 rows × 29 columns; exactly {FINAL_BEST_ROUND} frozen rounds; "
    "no early stopping and no tuning"
)
display(pd.DataFrame({"Frozen value": [
    PRE_HOLDOUT_FREEZE_RECORD["freeze_status"], FINAL_CHALLENGER_HYPERPARAMETERS,
    FINAL_BEST_ROUND, FINAL_DEVELOPMENT_FOLD_AUCS, final_selected["Mean AUC"],
    final_selected["AUC SD"], FINAL_DEVELOPMENT_FOLD_KS, final_selected["Mean KS"],
    final_selected["KS SD"], FULL_DEVELOPMENT_REFIT_STATUS, HOLDOUT_SEAL_STATUS,
]}, index=[
    "Freeze status", "Exact final hyperparameters", "Frozen best_round", "Fold AUCs",
    "Mean CV AUC", "AUC sample SD", "Fold KS", "Mean CV KS", "KS sample SD",
    "Full-Development refit", "Holdout status in Notebook 11",
]))


,Frozen value
Freeze status,FROZEN BEFORE HOLDOUT PREDICTIVE ACCESS
Exact final hyperparameters,"{'objective': 'binary:logistic', 'booster': 'g..."
Frozen best_round,54
Fold AUCs,"[0.7861789492974554, 0.7865445634442015, 0.779..."
Mean CV AUC,0.785879288
AUC sample SD,0.003556830
Fold KS,"[0.44946243823226106, 0.43559654548037713, 0.4..."
Mean CV KS,0.437782988
KS sample SD,0.009842402
Full-Development refit,"PASS — 24,000 rows × 29 columns; exactly 54 fr..."


## 14. Development conclusion and Notebook 12 handoff

Notebook 11 contains no Holdout predictive analysis. The frozen Primary and the single frozen Challenger may now be evaluated once on the same 6,000-row Holdout in Notebook 12. No feature, preprocessing, hyperparameter, round, or model-family decision is permitted after that opening.


In [15]:
assert PRE_HOLDOUT_FREEZE_RECORD["freeze_status"] == "FROZEN BEFORE HOLDOUT PREDICTIVE ACCESS"
assert HOLDOUT_SEAL_STATUS.startswith("SEALED")
assert len(final_joint) == projected_final_joint_count == 675
assert len(final_near) >= 1
assert FINAL_BEST_ROUND >= 1

stage5_development_result = {
    "status": "CHALLENGER_FROZEN_READY_FOR_HOLDOUT",
    "holdout_predictively_accessed": False,
    "holdout_seal_status": HOLDOUT_SEAL_STATUS,
    "train_hash": train_hash,
    "test_hash": test_hash,
    "baseline": json.loads(pd.Series(baseline_row).to_json(double_precision=15)),
    "phase_a": json.loads(phase_a.to_json(orient="records", double_precision=15)),
    "phase_a_winner": json.loads(phase_a_best.to_json(double_precision=15)),
    "phase_a_winner_se": phase_a_se,
    "phase_a_threshold": phase_a_threshold,
    "phase_a_near_best": json.loads(phase_a_near.to_json(orient="records", double_precision=15)),
    "phase_a_retained_depths": phase_a_retained_depths,
    "phase_a_retained_children": phase_a_retained_children,
    "phase_a_depth_profile": json.loads(phase_a_depth_profile.to_json(orient="records", double_precision=15)),
    "phase_a_child_profile": json.loads(phase_a_child_profile.to_json(orient="records", double_precision=15)),
    "phase_a_boundary_resolution": phase_a_boundary_resolution,
    "phase_b": json.loads(phase_b.to_json(orient="records", double_precision=15)),
    "phase_b_winner": json.loads(phase_b_best.to_json(double_precision=15)),
    "phase_b_winner_se": phase_b_se,
    "phase_b_threshold": phase_b_threshold,
    "phase_b_near_best": json.loads(phase_b_near.to_json(orient="records", double_precision=15)),
    "phase_b_retained_subsamples": phase_b_retained_subsamples,
    "phase_b_retained_colsamples": phase_b_retained_colsamples,
    "phase_b_profiles": {
        "subsample": json.loads(phase_b_subsample_profile.to_json(orient="records", double_precision=15)),
        "colsample_bytree": json.loads(phase_b_colsample_profile.to_json(orient="records", double_precision=15)),
    },
    "phase_b_boundary_result": "NO TRIGGER — negligible profile fluctuation relative to winner SE",
    "phase_c": json.loads(phase_c.to_json(orient="records", double_precision=15)),
    "phase_c_winner": json.loads(phase_c_best.to_json(double_precision=15)),
    "phase_c_winner_se": phase_c_se,
    "phase_c_threshold": phase_c_threshold,
    "phase_c_near_best": json.loads(phase_c_near.to_json(orient="records", double_precision=15)),
    "phase_c_retained_rates": phase_c_retained_rates,
    "phase_c_profile": json.loads(phase_c_profile.to_json(orient="records", double_precision=15)),
    "phase_c_boundary_result": "NO TRIGGER — negligible profile fluctuation relative to winner SE",
    "final_joint_candidates": final_joint_candidates,
    "final_joint_count": len(final_joint),
    "final_joint": json.loads(final_joint.to_json(orient="records", double_precision=15)),
    "final_raw_best": json.loads(final_raw_best.to_json(double_precision=15)),
    "final_best_se": final_best_se,
    "final_threshold": final_threshold,
    "final_near_best": json.loads(final_near.to_json(orient="records", double_precision=15)),
    "final_selected": json.loads(final_selected.to_json(double_precision=15)),
    "final_hyperparameters": FINAL_CHALLENGER_HYPERPARAMETERS,
    "final_best_round": FINAL_BEST_ROUND,
    "selection_reason": final_selection_reason,
    "pre_holdout_freeze_record": PRE_HOLDOUT_FREEZE_RECORD,
    "full_development_refit_status": FULL_DEVELOPMENT_REFIT_STATUS,
}
print("STAGE5_DEVELOPMENT_RESULT_JSON=" + json.dumps(stage5_development_result, ensure_ascii=False, sort_keys=True, default=float))
print("PASS — Challenger frozen and full-Development refit complete; Holdout remained predictively sealed throughout Notebook 11.")


STAGE5_DEVELOPMENT_RESULT_JSON={"baseline": {"AUC SD": 0.008242548250513, "Aggregate CV AUC at best round": 0.763078815243361, "Elapsed seconds": 3.09012770652771, "Fold 1 AUC": 0.768780883869467, "Fold 1 KS": 0.406067285274632, "Fold 2 AUC": 0.761869621205938, "Fold 2 KS": 0.396033408602443, "Fold 3 AUC": 0.75050192001725, "Fold 3 KS": 0.380399198338638, "Fold 4 AUC": 0.771964826049762, "Fold 4 KS": 0.407183212267958, "Fold 5 AUC": 0.762276825074387, "Fold 5 KS": 0.397871304936626, "KS SD": 0.010745796385923, "Label": "Untuned baseline", "Mean AUC": 0.763078815243361, "Mean KS": 0.397510881884059, "best_round": 100, "colsample_bytree": 1.0, "learning_rate": 0.3, "max_depth": 6, "min_child_weight": 1.0, "subsample": 1.0}, "final_best_round": 54, "final_best_se": 0.002351744949602327, "final_hyperparameters": {"booster": "gbtree", "colsample_bytree": 0.7, "eta": 0.1, "eval_metric": "auc", "gamma": 0.0, "max_depth": 4, "min_child_weight": 1.0, "nthread": 4, "objective": "binary:logistic"